## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-11-23-52-30 +0000,bbc,Democrat Scott Colom wins Mississippi Senate r...,https://www.bbc.co.uk/news/articles/cw7vppyr80...
1,2026-11-11-23-51-36 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/11/us/mis...
2,2026-11-11-23-50-00 +0000,wsj,"Naver Quarterly Profit Falls on Year, Missing ...",https://www.wsj.com/business/earnings/naver-qu...
3,2026-11-11-23-48-52 +0000,nyt,Soldier Shouts Toward Hegseth to Show ‘Respect...,https://www.nytimes.com/2026/11/11/us/politics...
4,2026-11-11-23-48-46 +0000,nypost,California sheriff’s deputy accused of repeate...,https://nypost.com/2026/11/11/us-news/san-dieg...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2465/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
167,trump,33
41,nvidia,19
257,war,18
292,house,17
38,huang,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
66,2026-11-11-22-20-00 +0000,wsj,Vice President JD Vance is considering face-to...,https://www.wsj.com/world/middle-east/jd-vance...,156
61,2026-11-11-22-25-23 +0000,nypost,Nvidia’s Jensen Huang also had alleged affair ...,https://nypost.com/2026/11/11/business/nvidias...,136
113,2026-11-11-21-13-00 +0000,wsj,"Democrats, fresh off sweeping midterm wins, ar...",https://www.wsj.com/politics/national-security...,125
281,2026-11-11-11-53-00 +0000,wsj,House Republicans saved Pete Hegseth and picke...,https://www.wsj.com/politics/policy/house-gop-...,121
408,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...,115


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
66,156,2026-11-11-22-20-00 +0000,wsj,Vice President JD Vance is considering face-to...,https://www.wsj.com/world/middle-east/jd-vance...
61,136,2026-11-11-22-25-23 +0000,nypost,Nvidia’s Jensen Huang also had alleged affair ...,https://nypost.com/2026/11/11/business/nvidias...
113,110,2026-11-11-21-13-00 +0000,wsj,"Democrats, fresh off sweeping midterm wins, ar...",https://www.wsj.com/politics/national-security...
49,76,2026-11-11-22-39-00 +0000,wsj,U.S. Treasury Secretary Scott Bessent was rece...,https://www.wsj.com/politics/national-security...
281,61,2026-11-11-11-53-00 +0000,wsj,House Republicans saved Pete Hegseth and picke...,https://www.wsj.com/politics/policy/house-gop-...
0,56,2026-11-11-23-52-30 +0000,bbc,Democrat Scott Colom wins Mississippi Senate r...,https://www.bbc.co.uk/news/articles/cw7vppyr80...
362,52,2026-11-11-05-12-41 +0000,nypost,Mayor Karen Bass reveals her political future ...,https://nypost.com/2026/11/11/us-news/outgoing...
383,52,2026-11-11-02-09-00 +0000,wsj,The Kennedy family is joining the efforts to b...,https://www.wsj.com/politics/policy/kennedy-fa...
277,44,2026-11-11-11-57-33 +0000,nypost,"Former NC Lt. Gov. Mark Robinson, who called h...",https://nypost.com/2026/11/11/us-news/former-n...
239,41,2026-11-11-14-40-48 +0000,nypost,Joe Biden claims he would have resigned if he ...,https://nypost.com/2026/11/11/us-news/joe-bide...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
